# Data Preparation

### Importing the libraries

In [1]:
import json
import os

### Loading dataset 

In [2]:
dataset_paths = {
    "train": "/kaggle/input/animal-detection-model/train/train",
    "test": "/kaggle/input/animal-detection-model/test",
    "validate": "/kaggle/input/animal-detection-model/valid"
}

### Loading data from all folders

In [4]:
dataset_info = {}

for folder, path in dataset_paths.items():
    images, annotations, categories = load_coco_annotations(folder, path)
    dataset_info[folder] = {"images": images, "annotations": annotations, "categories": categories}
    
    print(f"Loaded {len(images)} images and {len(annotations)} annotations from {folder}")
    print(f"Categories: {categories}")

Loaded 1115 images and 1359 annotations from train
Categories: {0: 'Animals', 1: 'Carnivorous', 2: 'Herbivorous'}
Loaded 67 images and 86 annotations from test
Categories: {0: 'Animals', 1: 'Carnivorous', 2: 'Herbivorous'}
Loaded 129 images and 160 annotations from validate
Categories: {0: 'Animals', 1: 'Carnivorous', 2: 'Herbivorous'}


# Model training and Building

### Importing the libraries

In [5]:
import torch
import torchvision
import torchvision.transforms as T
import os
import json
import torchvision.models.detection as models

from torch.utils.data import Dataset, DataLoader
from PIL import Image

### Preprocessing & augmentation transforms

In [6]:
transform = T.Compose([
    T.Resize((640, 640)),  
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Augmentation
    T.ToTensor(),  
])

### Loading the Dataset

In [7]:
dataset_folders = {
    "train": "/kaggle/input/animal-detection-model/train/train",
    "test": "/kaggle/input/animal-detection-model/test",
    "validate": "/kaggle/input/animal-detection-model/valid"
}

### Preprocessing images

In [8]:
def preprocess_images(folder_path):
    processed_images = []
    
    for img_file in os.listdir(folder_path):
        if img_file.endswith((".jpg", ".png", ".jpeg")):  
            img_path = os.path.join(folder_path, img_file)
            img = Image.open(img_path).convert("RGB")  
            img_transformed = transform(img)  # Apply transforms
            processed_images.append(img_transformed)

    return processed_images

In [9]:
train_images = preprocess_images(dataset_folders["train"])
test_images = preprocess_images(dataset_folders["test"])
val_images = preprocess_images(dataset_folders["validate"])

print(f"Processed {len(train_images)} train images, {len(test_images)} test images, {len(val_images)} validation images.")

Processed 1115 train images, 67 test images, 129 validation images.


### Loading Annotations

In [11]:
def load_coco_annotations(folder_path):
    json_path = os.path.join(folder_path, "_annotations.coco.json")
    with open(json_path, "r") as f:
        data = json.load(f)

    images = {img["id"]: img["file_name"] for img in data["images"]}
    annotations = data["annotations"]
    categories = {cat["id"]: cat["name"] for cat in data["categories"]}

    return images, annotations, categories

In [12]:
class AnimalDataset(Dataset):
    def __init__(self, folder_path, transforms=None):
        self.folder_path = folder_path
        self.transforms = transforms
        self.images, self.annotations, self.categories = load_coco_annotations(folder_path)

        self.image_annotations = {img_id: [] for img_id in self.images}
        for ann in self.annotations:
            self.image_annotations[ann["image_id"]].append(ann)

        # Filtering out images with no bounding boxes
        self.image_ids = [img_id for img_id in self.images if len(self.image_annotations[img_id]) > 0]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.folder_path, self.images[img_id])
        img = Image.open(img_path).convert("RGB")

        # Get bounding boxes & labels
        boxes = []
        labels = []
        for ann in self.image_annotations[img_id]:
            x, y, w, h = ann["bbox"]
            boxes.append([x, y, x + w, y + h])
            labels.append(ann["category_id"]) 

        # Converting to tensors
        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            img = self.transforms(img)

        return img, target

In [13]:
# Loading preprocessed dataset
train_dataset = AnimalDataset(dataset_folders["train"])
test_dataset = AnimalDataset(dataset_folders["test"])
val_dataset = AnimalDataset(dataset_folders["validate"])

# Print dataset sizes
print(f"Train dataset: {len(train_dataset)} images")
print(f"Test dataset: {len(test_dataset)} images")
print(f"Validation dataset: {len(val_dataset)} images")

Train dataset: 1101 images
Test dataset: 66 images
Validation dataset: 128 images


In [15]:
transform = T.Compose([
    T.Resize((640, 640)),
    T.ToTensor(),
])

In [16]:
train_dataset = AnimalDataset(dataset_folders["train"], transforms=transform)
val_dataset = AnimalDataset(dataset_folders["validate"], transforms=transform)

### Defining DataLoaders

In [17]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

In [18]:
model = models.fasterrcnn_resnet50_fpn(pretrained=True)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [19]:
# Modify the classifier for 2 classes (Carnivorous & Herbivorous)
num_classes = 3
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)


### Defining training parameters

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 10

In [21]:
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

Epoch 1/10, Loss: 53.4245
Epoch 2/10, Loss: 41.2457
Epoch 3/10, Loss: 33.8136
Epoch 4/10, Loss: 30.6737
Epoch 5/10, Loss: 26.4403
Epoch 6/10, Loss: 28.3823
Epoch 7/10, Loss: 22.6788
Epoch 8/10, Loss: 17.7383
Epoch 9/10, Loss: 16.9088
Epoch 10/10, Loss: 14.9030


In [22]:
# Save the trained model
torch.save(model.state_dict(), "animal_detection_model.pth")
print("Model saved as animal_detection_model.pth")

Model saved as faster_rcnn_animal.pth


# Model Evaluation

### Importing Libraries

In [79]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

### Loading trained model

In [81]:
model.load_state_dict(torch.load("/kaggle/working/faster_rcnn_animal.pth", weights_only=True))

<All keys matched successfully>

In [82]:
# Checking availability of GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Defining confidence threshold 
score_threshold = 0.5  

# Lists to store true and predicted values 
true_labels = []
pred_labels = []

# Run inference on validation dataset
with torch.no_grad():
    for images, targets in val_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)  

        for i, output in enumerate(outputs):
            # Get predicted labels with confidence above threshold
            pred_classes = output["labels"][output["scores"] > score_threshold].cpu().numpy()
            true_classes = targets[i]["labels"].cpu().numpy()

            # Ensure both lists have the same length
            min_length = min(len(true_classes), len(pred_classes))
            true_labels.extend(true_classes[:min_length])
            pred_labels.extend(pred_classes[:min_length])

### Converting to NumPy arrays

In [83]:
true_labels = np.array(true_labels)
pred_labels = np.array(pred_labels)

### Computing metrics

In [84]:
accuracy = accuracy_score(true_labels, pred_labels) * 100
precision = precision_score(true_labels, pred_labels, average="weighted") * 100
recall = recall_score(true_labels, pred_labels, average="weighted") * 100
conf_matrix = confusion_matrix(true_labels, pred_labels)

### Results

In [85]:
print(f"Model Accuracy: {accuracy:.2f}%")
print(f"Precision: {precision:.2f}%")
print(f"Recall: {recall:.2f}%")
print("Confusion Matrix:")
print(conf_matrix)

Model Accuracy: 84.14%
Precision: 84.81%
Recall: 84.14%
Confusion Matrix:
[[60  7]
 [16 62]]
